# A6b · Molecular fingerprints: from SMILES to bits, and what the bits cannot tell
**Series A: Machine Learning Basics for Chemists, Post 6b** &nbsp;|&nbsp; companion *primer* notebook

Section 2.7 of the notes gives a tutorial and assignment on **molecular fingerprints and multitask prediction**: SMILES strings become 1,024-bit circular (Morgan, ECFP) fingerprints with RDKit, similarity is measured with the Tanimoto coefficient, labels carry a mask for missing values, and a logistic regression and a small neural network are compared by ROC AUC. This primer rebuilds the **whole chain with NumPy only**: a SMILES parser, the circular fingerprint with its duplicate rule, the Tanimoto coefficient, the masked loss, regularized logistic regression, and a shared multitask network. It reproduces the facts that the instructor notebook printed with RDKit (active bits $1,2,4$; the most similar pairs; the masked loss $0.36355$) and the class counts behind its AUCs, and then asks what a 76-molecule exercise can and cannot show. Part II repeats the key steps with RDKit and scikit-learn.

> **Two parts.** **Part I** needs only NumPy, pandas and matplotlib and runs in well under a minute on a CPU; the 76 molecules are embedded, so no data file is needed. **Part II** needs RDKit (installed by the cell) and scikit-learn and compares every function with the libraries.
>
> **Reading the code.** Every section starts with a short **Stop and predict** question: answer it before running the cell.

| Section | What you do |
|---|---|
| 1 | a SMILES parser, hydrogens, and a check against RDKit's molecular weights |
| 2 | circular fingerprints from scratch; what they cannot see |
| 3 | the Tanimoto coefficient |
| 4 | labels, masks, and the masked loss |
| 5 | splits, logistic regression, and AUC as a count of pairs |
| 6 | how much does one split say? |
| 7 | overfitting and regularization (L2 and L1) |
| 8 | a shared multitask network |
| **Part II** | cross-check with RDKit and scikit-learn |

## 0. Setup and data

In [ ]:
import math, time, importlib.util, io
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True, linewidth=120)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")

In [ ]:
CSV = """
smiles,MolWt,LogP,HBD,HBA,demo_task_lipophilic,demo_task_hbond
C,16.043,0.6361,0,0,0.0,
CC,30.07,1.0262,0,0,0.0,0.0
CCC,44.096999999999994,1.4163,0,0,0.0,0.0
CCCC,58.123999999999995,1.8064,0,0,1.0,0.0
CCCCC,72.151,2.1965000000000003,0,0,1.0,0.0
CCCCCC,86.178,2.5866000000000007,0,0,1.0,0.0
CCCCCCC,100.205,2.976700000000002,0,0,1.0,0.0
CCCCCCCC,114.232,3.3668000000000022,0,0,1.0,
CCO,46.069,-0.0014000000000000123,1,1,0.0,0.0
CCCO,60.096,0.3887,1,1,0.0,0.0
CCCCO,74.12299999999999,0.7787999999999999,1,1,0.0,0.0
CC(C)O,60.096000000000004,0.38710000000000006,1,1,0.0,0.0
CO,32.042,-0.3915,1,1,0.0,0.0
CCOC,60.096000000000004,0.6527000000000001,0,1,0.0,0.0
CCOCC,74.123,1.0428000000000002,0,1,0.0,
CC(=O)O,60.05200000000001,0.09089999999999993,1,1,0.0,0.0
CCC(=O)O,74.079,0.481,1,1,0.0,0.0
CCCC(=O)O,88.10599999999998,0.8711,1,1,0.0,0.0
O=C(O)c1ccccc1,122.12299999999996,1.3848,1,1,0.0,0.0
COC(C)=O,74.07900000000001,0.17930000000000001,0,2,0.0,1.0
CC(N)=O,59.068,-0.5084,1,1,0.0,0.0
CCN,45.084999999999994,-0.03499999999999981,1,1,0.0,
CCCN,59.11199999999999,0.3551000000000001,1,1,0.0,0.0
CCCCN,73.139,0.7452000000000001,1,1,0.0,0.0
CN(C)C,59.111999999999995,0.17779999999999996,0,1,0.0,0.0
CNC(C)=O,73.095,-0.24770000000000003,1,1,0.0,0.0
CCS,62.137,0.9360999999999999,1,1,0.0,0.0
CCCS,76.16399999999999,1.3262,1,1,0.0,0.0
CS,48.11000000000001,0.546,1,1,0.0,
CCCl,64.515,1.2451,0,0,0.0,0.0
CCCCl,78.542,1.6352,0,0,1.0,0.0
CCBr,108.966,1.4012,0,0,0.0,0.0
CCF,48.06,0.9758,0,0,0.0,0.0
CCl,50.488,0.855,0,0,0.0,0.0
ClCCl,84.93299999999999,1.4215,0,0,0.0,0.0
ClC(Cl)Cl,119.37800000000001,1.9864,0,0,1.0,
c1ccccc1,78.11399999999999,1.6866,0,0,1.0,0.0
Cc1ccccc1,92.14099999999999,1.99502,0,0,1.0,0.0
CCc1ccccc1,106.16799999999996,2.2490000000000006,0,0,1.0,0.0
Oc1ccccc1,94.11299999999999,1.3921999999999999,1,1,0.0,0.0
COc1ccccc1,108.13999999999997,1.6952,0,1,1.0,0.0
Nc1ccccc1,93.12899999999999,1.2688000000000001,1,1,0.0,0.0
O=[N+]([O-])c1ccccc1,123.11099999999996,1.5948,0,2,1.0,
Fc1ccccc1,96.10399999999998,1.8256999999999999,0,0,1.0,0.0
Clc1ccccc1,112.55899999999997,2.34,0,0,1.0,0.0
Brc1ccccc1,157.01,2.4491000000000005,0,0,1.0,0.0
c1ccncc1,79.10199999999998,1.0816,0,1,0.0,0.0
c1ccoc1,68.07499999999999,1.2795999999999998,0,1,0.0,0.0
c1ccsc1,84.14299999999999,1.7480999999999998,0,1,1.0,0.0
c1cc[nH]c1,67.091,1.0147,1,0,0.0,
c1ccc2ccccc2c1,128.17399999999995,2.839800000000001,0,0,1.0,0.0
c1ccc(-c2ccccc2)cc1,154.21199999999996,3.353600000000001,0,0,1.0,0.0
CC(C)C,58.123999999999995,1.6622999999999999,0,0,1.0,0.0
CC(C)(C)C,72.151,2.0524,0,0,1.0,0.0
CC(C)CO,74.12299999999999,0.6347,1,1,0.0,0.0
CC(C)C(=O)O,88.10599999999998,0.727,1,1,0.0,0.0
OCCO,62.068000000000005,-1.0290000000000001,2,2,0.0,
OCCCO,76.095,-0.6389,2,2,0.0,1.0
NCCO,61.083999999999996,-1.0626000000000002,2,2,0.0,1.0
NCCN,60.09999999999999,-1.0962,2,2,0.0,1.0
CC(C)=O,58.08,0.5952999999999999,0,1,0.0,0.0
CCC(C)=O,72.107,0.9854,0,1,0.0,0.0
CC#N,41.053,0.52988,0,1,0.0,0.0
CCC#N,55.07999999999999,0.91998,0,1,0.0,
C1CCCCC1,84.162,2.3406000000000002,0,0,1.0,0.0
C1CCOCC1,86.134,1.1869,0,1,0.0,0.0
C1CCNCC1,85.15,0.7599,1,1,0.0,0.0
O=C1CCCCC1,98.14500000000001,1.5196,0,1,1.0,0.0
OC1CCCCC1,100.161,1.3114,1,1,0.0,0.0
CC(=O)Oc1ccccc1C(=O)O,180.15899999999996,1.3101,1,3,0.0,1.0
CC(C)Cc1ccc(C(C)C(=O)O)cc1,206.28499999999997,3.073200000000001,1,1,1.0,
Cn1c(=O)c2c(ncn2C)n(C)c1=O,194.19399999999996,-1.0293,0,6,0.0,1.0
CC(=O)Nc1ccc(O)cc1,151.165,1.3505999999999998,2,2,0.0,1.0
CN1CCC(O)CC1,115.17599999999999,0.07290000000000002,1,2,0.0,1.0
NC(=O)c1ccccc1,121.13899999999995,0.7855000000000001,1,1,0.0,0.0
O=C(O)c1ccc(O)cc1,138.12199999999999,1.0903999999999998,2,2,0.0,1.0
"""

In [ ]:
df = pd.read_csv(io.StringIO(CSV)); sm = df.smiles.tolist(); N = len(df)
Y = df[["demo_task_lipophilic", "demo_task_hbond"]].to_numpy(float); Wm = np.isfinite(Y).astype(float)             # labels and the mask: 1 = known, 0 = missing
print(f"{N} molecules, columns {list(df.columns)}")
print("labels known per task:", Wm.sum(0).astype(int), "| positives:", np.nansum(Y, 0).astype(int))

## 1. From SMILES to a molecular graph
A **SMILES** string writes a molecule as text; two different strings can describe the same molecule, and a model needs numbers of a fixed size. The first step is to parse the string into **atoms and bonds** and to count the hydrogens that the string leaves implicit: each atom has a default valence (C 4, N 3, O 2, S 2, halogens 1), and the missing bonds are filled with hydrogens; an aromatic bond counts 1, and an aromatic carbon (or a pyridine-type nitrogen) has one extra $\pi$ bond. A check that the parser is right: the molecular weight from the atoms and hydrogens must equal the weight that RDKit wrote in the CSV.
**Stop and predict:** how many hydrogens does the nitrogen of pyrrole `c1cc[nH]c1` have, and the carbon of benzene?

In [ ]:
import re
MASS = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "S": 32.06, "F": 18.998, "Cl": 35.45, "Br": 79.904}
Z = {"H": 1, "C": 6, "N": 7, "O": 8, "F": 9, "S": 16, "Cl": 17, "Br": 35}
VAL = {"C": 4, "N": 3, "O": 2, "S": 2, "F": 1, "Cl": 1, "Br": 1}
TOKEN = re.compile(r"\[[^\]]+\]|Cl|Br|[BCNOSPFI]|[bcnosp]|\(|\)|=|#|-|:|\/|\\|\d|%\d\d|\.")
def parse_smiles(smi):
    """SMILES -> atoms (element, aromatic, charge, explicit_H or None) and bonds (i, j, order); order 1.5 marks an aromatic bond"""
    atoms, bonds, stack, prev, pending, rings = [], [], [], None, None, {}
    for tok in TOKEN.findall(smi):
        if tok == "(": stack.append(prev)
        elif tok == ")": prev = stack.pop()
        elif tok in "-=#:": pending = {"-": 1, "=": 2, "#": 3, ":": 1.5}[tok]
        elif tok.isdigit():
            if tok in rings: j, o = rings.pop(tok); bonds.append((prev, j, o if o is not None else (pending or (1.5 if atoms[prev][1] and atoms[j][1] else 1)))); pending = None
            else: rings[tok] = (prev, pending); pending = None
        else:
            if tok.startswith("["):
                m = re.match(r"\[([A-Za-z][a-z]?)(H\d?)?([+-]\d?)?\]", tok); el = m.group(1); arom = el.islower(); el = el.capitalize()
                nh = 0 if not m.group(2) else (1 if m.group(2) == "H" else int(m.group(2)[1:])); ch = 0 if not m.group(3) else (int(m.group(3)[1:] or 1) * (1 if m.group(3)[0] == "+" else -1)); atoms.append((el, arom, ch, nh))
            else: arom = tok.islower(); atoms.append((tok.capitalize(), arom, 0, None))
            i = len(atoms) - 1
            if prev is not None: bonds.append((prev, i, pending if pending else (1.5 if atoms[prev][1] and arom else 1)))
            prev, pending = i, None
    return atoms, bonds
def add_hydrogens(atoms, bonds):
    """implicit hydrogens from the default valences (aromatic bonds count 1, plus one pi bond for carbon and pyridine-type nitrogen)"""
    used = [0.0] * len(atoms); nb = [[] for _ in atoms]
    for i, j, o in bonds: nb[i].append((j, o)); nb[j].append((i, o)); w = 1 if o == 1.5 else o; used[i] += w; used[j] += w
    H = []
    for i, (el, arom, ch, nh) in enumerate(atoms):
        if nh is not None: H.append(nh); continue                                             # bracket atoms: the hydrogens are written
        v = VAL[el] + (1 if (el == "N" and ch == 1) else 0) - (1 if (el == "O" and ch == 1) else 0)
        u = used[i]
        if arom and el in ("C", "N") and u < v: u += 1                                          # the pi bond of an aromatic atom
        H.append(int(max(0, v - u)))
    return H, nb
def ring_atoms(n, bonds):
    """atoms and bonds that lie on a ring: a bond is a ring bond if its ends stay connected without it"""
    adj = {i: [] for i in range(n)}
    for k, (i, j, o) in enumerate(bonds): adj[i].append((j, k)); adj[j].append((i, k))
    def connected(a, b, skip):
        seen, st = {a}, [a]
        while st:
            x = st.pop()
            if x == b: return True
            for y, k in adj[x]:
                if k != skip and y not in seen: seen.add(y); st.append(y)
        return False
    rb = [connected(i, j, k) for k, (i, j, o) in enumerate(bonds)]; ra = set()
    for k, (i, j, o) in enumerate(bonds):
        if rb[k]: ra.update((i, j))
    return ra, rb
def mol_weight(atoms, H): return sum(MASS[a[0]] for a in atoms) + sum(H) * MASS["H"]
def hash32(values):
    h = 2166136261
    for v in values: h = ((h ^ (int(v) & 0xFFFFFFFF)) * 16777619) & 0xFFFFFFFF
    return h
def circular_environments(smi, radius=2, salt=0):
    """Morgan-style circular fingerprint: the list of distinct environment identifiers (before folding); the duplicate rule drops an environment that covers the same bonds as an earlier one"""
    atoms, bonds = parse_smiles(smi); H, nb = add_hydrogens(atoms, bonds); ra, rb = ring_atoms(len(atoms), bonds); n = len(atoms)
    inv = [hash32([salt, Z[a[0]], len(nb[i]), H[i], a[2], int(i in ra)]) for i, a in enumerate(atoms)]; ids = set(inv); seen = [tuple()]; dead = [False] * n
    bset = [frozenset() for _ in range(n)]; bidx = {}
    for k, (i, j, o) in enumerate(bonds): bidx[(i, j)] = bidx[(j, i)] = k
    for layer in range(1, radius + 1):
        new_inv, new_bs, cand = list(inv), list(bset), []
        for i in range(n):
            if dead[i]: continue
            ns = sorted((int(o * 2), inv[j]) for j, o in nb[i]); new_inv[i] = hash32([salt, layer, inv[i]] + [x for p in ns for x in p])
            s = set(bset[i])
            for j, o in nb[i]: s |= set(bset[j]); s.add(bidx[(i, j)])
            new_bs[i] = frozenset(s); cand.append((tuple(sorted(new_bs[i])), new_inv[i], i))
        for bs, iv, i in sorted(cand):
            if bs not in seen: ids.add(iv); seen.append(bs)
            else: dead[i] = True
        inv, bset = new_inv, new_bs
    return sorted(ids)
def fold(env_ids, size=1024): x = np.zeros(size); x[[e % size for e in env_ids]] = 1; return x

In [ ]:
at, bd = parse_smiles("c1cc[nH]c1"); H, nb_ = add_hydrogens(at, bd); print("pyrrole: atoms", [a[0] for a in at], "hydrogens", H)
at, bd = parse_smiles("c1ccccc1"); H, _ = add_hydrogens(at, bd); print("benzene: hydrogens per atom", H, "| bonds", len(bd), "all aromatic:", all(o == 1.5 for _, _, o in bd))
mw = []
for s in sm: a_, b_ = parse_smiles(s); h_, _ = add_hydrogens(a_, b_); mw.append(mol_weight(a_, h_))
d_mw = np.abs(np.array(mw) - df.MolWt.values)
print(f"molecular weight from our atoms and hydrogens against RDKit's (column MolWt): largest difference {d_mw.max():.4f} over {N} molecules (worst: {sm[int(d_mw.argmax())]}), none above 0.02: {bool((d_mw < 0.02).all())}")
def formula(s):
    a_, b_ = parse_smiles(s); h_, _ = add_hydrogens(a_, b_); c = {"H": sum(h_)}
    for x in a_: c[x[0]] = c.get(x[0], 0) + 1
    order = ["C", "H"] + sorted(e for e in c if e not in ("C", "H")); return "".join(e + (str(c[e]) if c[e] > 1 else "") for e in order if e in c)
print("formulas from the parsed graphs:", {s: formula(s) for s in ("CCO", "c1ccccc1", "CC(=O)O", "O=[N+]([O-])c1ccccc1", "Cn1c(=O)c2c(ncn2C)n(C)c1=O")})
assert H == [1, 1, 1, 1, 1, 1] and (d_mw < 0.02).all() and formula('CCO') == 'C2H6O' and formula('Cn1c(=O)c2c(ncn2C)n(C)c1=O') == 'C8H10N4O2'

## 2. Circular (Morgan, ECFP) fingerprints
The idea: describe each atom by its **environment** and collect the environments that occur. Round 0: an atom's identifier comes from its own properties (element, number of heavy neighbors, hydrogens, charge, ring membership). Round $r$: the new identifier is a **hash** of the atom's previous identifier and of its neighbors' (with the bond types), so that it encodes everything within $r$ bonds. With radius 2 (called ECFP4 by the diameter convention) every atom contributes three environments. All identifiers go into a bit vector of fixed length by taking the hash modulo 1,024, so molecules of any size give vectors of the same size, and **different environments can land on the same bit** (a collision).
A detail decides how many bits are on: an environment that covers *exactly the same bonds* as an earlier one (for ethane, the radius-2 environment of each carbon covers the single C--C bond, like the radius-1 one) is a duplicate and is dropped; a radius-1 environment of an isolated atom covers no bond at all, like radius 0.
**Stop and predict:** the number of active bits of methane, ethane and propane, and the fingerprints of heptane and octane.

In [ ]:
envs = [circular_environments(s) for s in sm]; X = np.stack([fold(e) for e in envs])              # 76 molecules x 1024 bits
print("active bits of the first three molecules (C, CC, CCC):", [len(e) for e in envs[:3]], "| the instructor notebook, with RDKit, printed 1, 2 and 4")
cnt = dict(zip(sm, map(len, envs))); print("active bits of hexane, heptane, octane:", cnt["CCCCCC"], cnt["CCCCCCC"], cnt["CCCCCCCC"])
alln = sorted(set().union(*map(set, envs))); used = int((X.sum(0) > 0).sum())
print(f"over the {N} molecules: {len(alln)} distinct environments; active bits per molecule: min {int(X.sum(1).min())}, median {int(np.median(X.sum(1)))}, max {int(X.sum(1).max())}")
print(f"folded into 1024 bits they occupy {used} positions: {len(alln) - used} environments share a bit with another ({100 * (len(alln) - used) / len(alln):.0f} %)")
once = int((X.sum(0) == 1).sum()); twice = int((X.sum(0) == 2).sum()); print(f"of the {used} bits in use, {once} are switched on by exactly one molecule and {twice} by exactly two: most bits are rare, and with 42 training molecules a rare bit identifies a molecule more than a property")
same = {}
for s, e in zip(sm, envs): same.setdefault(tuple(e), []).append(s)
print("distinct fingerprints:", len(same), "of", N, "| molecules with an identical fingerprint:", [g for g in same.values() if len(g) > 1])
assert once > 0.5 * used and [len(e) for e in envs[:3]] == [1, 2, 4] and (cnt["CCCCCC"], cnt["CCCCCCC"], cnt["CCCCCCCC"]) == (7, 8, 8) and len(alln) - used > 20 and len(same) == 75

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6)); ax[0].imshow(X[:25, :125], aspect="auto", interpolation="nearest", cmap="Greys"); ax[0].set(title="(a) 25 fingerprints, first 125 bits", xlabel="bit", ylabel="molecule")
ax[1].hist(X.sum(1), bins=np.arange(0, 28, 2), color=COLORS["blue"]); ax[1].set(title="(b) active bits per molecule", xlabel="active bits", ylabel="molecules")
ax[2].hist(X.sum(0)[X.sum(0) > 0], bins=np.arange(0.5, 40, 1), color=COLORS["orange"]); ax[2].set(title="(c) molecules per used bit", xlabel="molecules that switch the bit on", ylabel="bits"); plt.tight_layout(); plt.show()

## 3. The Tanimoto coefficient
For two fingerprints with $a$ and $b$ active bits and $c$ bits in common, $T=c/(a+b-c)$: the overlap divided by the union. It is $1$ for identical fingerprints and $0$ for disjoint ones. It measures the overlap of *structural environments*, not of biological behavior.
**Stop and predict:** the three most similar pairs of the 76 molecules.

In [ ]:
tanimoto = lambda p, q: len(p & q) / len(p | q)
print("toy example [1,1,0,0] and [1,0,1,0]:", tanimoto({0, 1}, {0, 2}), "(= 1/3) | methane and ethane:", tanimoto(set(envs[0]), set(envs[1])), "(RDKit: 0.0)")
fps = [set(e) for e in envs]; pairs = sorted(((tanimoto(fps[i], fps[j]), sm[i], sm[j]) for i in range(N) for j in range(i + 1, N)), reverse=True)
print(pd.DataFrame([(round(t, 3), a_, b_) for t, a_, b_ in pairs[:3]], columns=["similarity", "SMILES 1", "SMILES 2"]).to_string(index=False)); print("the instructor notebook printed: 1.000 CCCCCCC CCCCCCCC; 0.875 CCCCCC CCCCCCCC; 0.875 CCCCCC CCCCCCC")
j0 = sm.index("CCCCCCC"); print(f"heptane and octane have the same fingerprint, but their LogP is {df.LogP[j0]:.3f} and {df.LogP[sm.index('CCCCCCCC')]:.3f} and their weights {df.MolWt[j0]:.1f} and {df.MolWt[sm.index('CCCCCCCC')]:.1f}: with radius 2 a fingerprint cannot count a chain beyond seven carbons")
assert pairs[0][1:] == ("CCCCCCC", "CCCCCCCC") and pairs[0][0] == 1.0 and abs(pairs[1][0] - 0.875) < 1e-12 and {pairs[1][2], pairs[2][2]} == {"CCCCCCCC", "CCCCCCC"}

## 4. Labels, masks, and the masked loss
The CSV has two labels, `demo_task_lipophilic` and `demo_task_hbond`; the second is missing for 11 molecules. A **mask** $w_{it}$ is $1$ where the label is known and $0$ where it is not; a missing label is *not* a negative, so it must not enter the loss: $\mathcal L=\sum_{i,t}w_{it}\,\mathrm{BCE}(y_{it},p_{it})\big/\sum_{i,t}w_{it}$. The notebook says that the labels are computed from RDKit descriptors, not measured. The CSV contains the descriptors, so we can find the rules.
**Stop and predict:** which descriptor and which threshold produce each label, and which molecules have a missing label?

In [ ]:
lip, hb = df.demo_task_lipophilic.values, df.demo_task_hbond.values
print(f"lipophilic = 1 exactly when LogP >= 1.5: smallest LogP among positives {df.LogP[lip == 1].min():.4f}, largest among negatives {df.LogP[lip == 0].max():.4f}; rule reproduces all {N} labels: {bool(((df.LogP.values >= 1.5) == (lip == 1)).all())}")
known = ~np.isnan(hb); print(f"hbond = 1 exactly when HBA >= 2: the rule reproduces all {int(known.sum())} known labels: {bool(((df.HBA.values[known] >= 2) == (hb[known] == 1)).all())} | positives {int(hb[known].sum())} of {int(known.sum())}")
miss = np.where(~known)[0]; print("rows with a missing hbond label:", miss.tolist(), "| every 7th molecule:", bool((miss == np.arange(0, N, 7)).all()))
print("the labels are exact threshold functions of single descriptors, and the missing labels follow a fixed pattern unrelated to the label")
assert ((df.LogP.values >= 1.5) == (lip == 1)).all() and ((df.HBA.values[known] >= 2) == (hb[known] == 1)).all() and (miss == np.arange(0, N, 7)).all()
tg = np.array([[1., np.nan], [0., 1.]]); wt = np.isfinite(tg).astype(float); pr = np.array([[0.8, 0.1], [0.3, 0.6]]); t0 = np.nan_to_num(tg)
bce = -(t0 * np.log(pr + 1e-10) + (1 - t0) * np.log(1 - pr + 1e-10)); masked = (bce * wt).sum() / wt.sum()
print(f"masked loss of the instructor's example: {masked:.5f} (by hand: (-ln 0.8 - ln 0.7 - ln 0.6)/3 = {(-math.log(.8) - math.log(.7) - math.log(.6)) / 3:.5f}) | unmasked, with the missing label read as 0: {bce.mean():.5f}")
z = 0.3 * 1 + (-0.7) * 0 - 0.1; print(f"the neuron of the notebook: z = {z:.1f}, ReLU(z) = {max(0, z):.1f}")
assert abs(masked - 0.36355) < 1e-5 and abs(z - 0.2) < 1e-12

## 5. Splits, logistic regression, and AUC as a count of pairs
The notebook splits the 76 molecules 75/25 into train and test, and the train part again 75/25 into *fit* (42) and *validation* (15), with the seed 32. We rebuild scikit-learn's split exactly, check that it gives the class counts implied by the notebook's majority-baseline accuracies, fit an $L_2$-regularized **logistic regression** (the same objective as scikit-learn's default: $C\sum\text{logloss}+\tfrac12\|w\|^2$, the intercept free) by Newton's method, and compare the AUCs. An AUC is the fraction of (positive, negative) pairs in which the positive molecule gets the higher score, so with few molecules it can only take a few values.
**Stop and predict:** how many positive and negative molecules are in the test set of each task, and what values can the AUC take?

In [ ]:
def sk_split(idx, test_size, seed):
    n = len(idx); n_test = int(np.ceil(test_size * n)); perm = np.random.RandomState(seed).permutation(n); return idx[perm[n_test:]], idx[perm[:n_test]]
idx_train, idx_test = sk_split(np.arange(N), .25, 32); idx_fit, idx_valid = sk_split(idx_train, .25, 32)
print("sizes fit / validation / test:", len(idx_fit), len(idx_valid), len(idx_test), "(the notebook prints 42, 15, 19)")
rows = []
for j, t in enumerate(("lipophilic", "hbond")):
    for nm, ix in (("fit", idx_fit), ("validation", idx_valid), ("test", idx_test)):
        kk = ix[Wm[ix, j] == 1]; y_ = Y[kk, j]; rows.append((t, nm, len(kk), int(y_.sum()), int(len(y_) - y_.sum()), max(y_.mean(), 1 - y_.mean())))
cls = pd.DataFrame(rows, columns=["task", "split", "n", "positives", "negatives", "majority accuracy"]); print(cls.round(3).to_string(index=False))
print("the notebook's majority-baseline accuracies: lipophilic validation 0.800, test 0.789; hbond validation 0.857, test 0.875")
assert np.allclose(cls.loc[[1, 2, 4, 5], "majority accuracy"], [0.8, 0.789, 0.857, 0.875], atol=1e-3)

In [ ]:
def logreg_l2(Xa, y, C=1.0, iters=60):
    n, p = Xa.shape; on = np.where(Xa.any(0))[0]; A = np.c_[Xa[:, on], np.ones(n)]; q = len(on); v = np.zeros(q + 1); pen = np.r_[np.ones(q), 0.0]     # a bit that is off in every training molecule keeps the weight 0
    for _ in range(iters):
        pr_ = 1 / (1 + np.exp(-(A @ v))); g = C * A.T @ (pr_ - y) + pen * v; Hs = C * (A.T * (pr_ * (1 - pr_))) @ A + np.diag(pen) + 1e-10 * np.eye(q + 1); step = np.linalg.solve(Hs, g); v -= step
        if np.abs(step).max() < 1e-10: break
    w = np.zeros(p + 1); w[on] = v[:-1]; w[-1] = v[-1]; return w
def auc(y, s):
    pos, neg = s[y == 1], s[y == 0]; return ((pos[:, None] > neg[None, :]).sum() + 0.5 * (pos[:, None] == neg[None, :]).sum()) / (len(pos) * len(neg))
score = lambda Xa, w: np.c_[Xa, np.ones(len(Xa))] @ w
mine = {}
for j, t in enumerate(("lipophilic", "hbond")):
    fit = idx_fit[Wm[idx_fit, j] == 1]; w_ = logreg_l2(X[fit], Y[fit, j])
    for nm, ix in (("validation", idx_valid), ("test", idx_test)):
        kk = ix[Wm[ix, j] == 1]; s_ = score(X[kk], w_); mine[(t, nm)] = (auc(Y[kk, j], s_), ((s_ > 0) == Y[kk, j]).mean(), int(Y[kk, j].sum() * (len(kk) - Y[kk, j].sum())))
nbk = {("lipophilic", "validation"): 0.972, ("lipophilic", "test"): 0.817, ("hbond", "validation"): 1.000, ("hbond", "test"): 0.893}
out = pd.DataFrame([(t, nm, v[2], v[0], round(v[0] * v[2]), nbk[(t, nm)], round(nbk[(t, nm)] * v[2])) for (t, nm), v in mine.items()], columns=["task", "split", "pairs", "our AUC", "our pairs ranked right", "notebook AUC", "notebook pairs right"])
print(out.round(3).to_string(index=False))
print("the notebook's AUCs are exact fractions of the pairs (35/36, 49/60, 24/24, 25/28); ours differ by one pair where the hash collisions differ, because only 1024-bit folding is involved (Section 6)")
assert all(abs(r["notebook AUC"] * r["pairs"] - round(r["notebook AUC"] * r["pairs"])) < 0.05 for _, r in out.iterrows()) and all(abs(r["our AUC"] * r["pairs"] - round(r["our AUC"] * r["pairs"])) < 1e-9 for _, r in out.iterrows())

## 6. How much does one split say?
Three sources of variation hide behind a single table of AUCs. **(a) The hash:** a different hash function folds the same 278 environments into different bits, with different collisions. **(b) The split:** which molecules are in the test set. **(c) The representation:** a one-number descriptor against 1,024 bits.
**Stop and predict:** since both labels are thresholds of one descriptor, what AUC does that descriptor give on the test set?

In [ ]:
def split_aucs(Xm, tr_idx, te_idx, C=1.0):
    out = []
    for j in range(2):
        fit = tr_idx[Wm[tr_idx, j] == 1]; te = te_idx[Wm[te_idx, j] == 1]
        if len(np.unique(Y[fit, j])) < 2 or len(np.unique(Y[te, j])) < 2: out.append(np.nan); continue
        out.append(auc(Y[te, j], score(Xm[te], logreg_l2(Xm[fit], Y[fit, j], C))))
    return out
t0 = time.perf_counter(); salts = np.array([split_aucs(np.stack([fold(circular_environments(s, salt=sl)) for s in sm]), idx_fit, idx_test) for sl in range(30)])
col = {e: i for i, e in enumerate(alln)}; Xu = np.zeros((N, len(alln)))
for i, e in enumerate(envs): Xu[i, [col[x] for x in e]] = 1
print(f"(a) the same split, 30 different hash functions (1024 bits): test AUC lipophilic {salts[:, 0].mean():.3f} +- {salts[:, 0].std():.3f} (range {salts[:, 0].min():.3f} to {salts[:, 0].max():.3f}); hbond {salts[:, 1].mean():.3f} +- {salts[:, 1].std():.3f} (range {salts[:, 1].min():.3f} to {salts[:, 1].max():.3f})")
print("    without folding (one column per environment, no collisions):", np.round(split_aucs(Xu, idx_fit, idx_test), 3), "| the notebook's RDKit values 0.817 and 0.893 lie inside these ranges:", bool(salts[:, 0].min() <= 0.817 <= salts[:, 0].max() and salts[:, 1].min() <= 0.893 <= salts[:, 1].max()))
rg = np.random.default_rng(0); R = []
for _ in range(200): p_ = rg.permutation(N); R.append(split_aucs(X, p_[19:], p_[:19]))
R = np.array(R); ok = np.isfinite(R[:, 1])
print(f"(b) 200 random 75/25 splits: lipophilic test AUC {np.nanmean(R[:, 0]):.3f} +- {np.nanstd(R[:, 0]):.3f} (5th to 95th percentile {np.nanpercentile(R[:, 0], 5):.3f} to {np.nanpercentile(R[:, 0], 95):.3f}); hbond: {ok.sum()} of 200 splits have both classes in the test set, AUC {np.nanmean(R[:, 1]):.3f} +- {np.nanstd(R[:, 1]):.3f} ({np.nanpercentile(R[:, 1], 5):.3f} to {np.nanpercentile(R[:, 1], 95):.3f})")
te0, te1 = idx_test[Wm[idx_test, 0] == 1], idx_test[Wm[idx_test, 1] == 1]
print(f"(c) the descriptor itself as the score: LogP gives AUC {auc(Y[te0, 0], df.LogP.values[te0]):.3f} for lipophilic, HBA gives {auc(Y[te1, 1], df.HBA.values[te1]):.3f} for hbond (the labels are thresholds of them)")
order = np.argsort(df.MolWt.values); sz = split_aucs(X, order[:57], order[57:])
print(f"(d) train on the 57 lightest molecules, test on the 19 heaviest (MolWt above {df.MolWt.values[order[57]]:.1f}): AUC lipophilic {sz[0]:.3f}, hbond {sz[1]:.3f}; positives in that test set: {int(Y[order[57:], 0].sum())} (lipophilic), {int(np.nansum(Y[order[57:], 1]))} of {int(Wm[order[57:], 1].sum())} (hbond) [{time.perf_counter() - t0:.0f} s]")
assert auc(Y[te0, 0], df.LogP.values[te0]) == 1.0 and auc(Y[te1, 1], df.HBA.values[te1]) == 1.0 and np.nanstd(R[:, 0]) > 0.05 and salts[:, 0].std() > 0.02

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.7)); ax[0].hist(R[:, 0][np.isfinite(R[:, 0])], bins=np.linspace(0.4, 1.0, 17), color=COLORS["blue"], alpha=0.8); ax[0].axvline(0.817, color=COLORS["red"], ls="--", label="the notebook's split (0.817)")
ax[0].set(xlabel="test AUC, lipophilic", ylabel="splits of 200", title="(a) 19 test molecules, 200 splits"); ax[0].legend(fontsize=8)
ax[1].hist(salts[:, 0], bins=np.linspace(0.7, 0.95, 11), color=COLORS["orange"], alpha=0.8); ax[1].axvline(0.817, color=COLORS["red"], ls="--", label="the notebook (0.817)"); ax[1].set(xlabel="test AUC, lipophilic", ylabel="hash functions of 30", title="(b) one split, 30 hash functions"); ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()

## 7. Overfitting and regularization
With 42 training molecules and 1,024 bits there are far more parameters than data: the model can fit the training set exactly. The strength of the penalty decides how much of that freedom is used: $C$ large means weak regularization. We sweep $C$ for the $L_2$ penalty (shrinks all weights) and for the $L_1$ penalty (sets most weights to exactly zero, solved by the proximal gradient method FISTA), and watch the training, validation and test AUC.
**Stop and predict:** which value of $C$ will the validation set choose, and is it the best for the test set?

In [ ]:
Cs = (1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100, 1000); fit0, va0, te0_ = idx_fit, idx_valid, idx_test; rows = []
for C in Cs:
    w_ = logreg_l2(X[fit0], Y[fit0, 0], C); f_ = lambda ix: score(X[ix], w_)
    rows.append((C, auc(Y[fit0, 0], f_(fit0)), auc(Y[va0, 0], f_(va0)), auc(Y[te0_, 0], f_(te0_)), ((f_(fit0) > 0) == Y[fit0, 0]).mean(), np.abs(w_[:-1]).sum()))
reg = pd.DataFrame(rows, columns=["C", "fit AUC", "validation AUC", "test AUC", "fit accuracy", "sum |w|"]); print("L2 penalty, lipophilic task:\n" + reg.round(3).to_string(index=False))
best_v = reg.loc[reg["validation AUC"].idxmax(), "C"]; print(f"\nvalidation (15 molecules, 3 positives) picks C = {best_v:g} (first of the ties); the best test AUC is {reg['test AUC'].max():.3f} at C = {reg.loc[reg['test AUC'].idxmax(), 'C']:g}; the test AUC at the validation's choice is {reg.loc[reg['C'] == best_v, 'test AUC'].item():.3f}")
assert reg.loc[reg["C"] == 1, "fit accuracy"].item() == 1.0 and reg["sum |w|"].is_monotonic_increasing and reg.loc[reg["C"] == 1, "validation AUC"].item() > 0.95

In [ ]:
def logreg_l1(Xa, y, C=1.0, iters=3000):
    n, p = Xa.shape; A = np.c_[Xa, np.ones(n)]; L = C * 0.25 * np.linalg.eigvalsh(A @ A.T).max() + 1e-12; w = np.zeros(p + 1); z = w.copy(); t = 1.0
    for _ in range(iters):
        g = C * A.T @ (1 / (1 + np.exp(-(A @ z))) - y); wn = z - g / L; wn[:-1] = np.sign(wn[:-1]) * np.maximum(np.abs(wn[:-1]) - 1 / L, 0)
        tn = (1 + np.sqrt(1 + 4 * t * t)) / 2; z = wn + ((t - 1) / tn) * (wn - w); w, t = wn, tn
    return w
rows = []
for C in (0.05, 0.1, 0.3, 1, 3, 10):
    w_ = logreg_l1(X[fit0], Y[fit0, 0], C); f_ = lambda ix: score(X[ix], w_); rows.append((C, int((np.abs(w_[:-1]) > 1e-8).sum()), ((f_(fit0) > 0) == Y[fit0, 0]).mean(), auc(Y[va0, 0], f_(va0)), auc(Y[te0_, 0], f_(te0_))))
l1 = pd.DataFrame(rows, columns=["C", "non-zero weights (of 1024)", "fit accuracy", "validation AUC", "test AUC"]); print("L1 penalty, lipophilic task:\n" + l1.round(3).to_string(index=False))
w1 = logreg_l1(X[fit0], Y[fit0, 0], 1.0); top = np.argsort(-np.abs(w1[:-1]))[:4]; env_of_bit = {}
for s, e in zip(sm, envs):
    for x in e: env_of_bit.setdefault(x % 1024, set()).add(s)
print("\nthe 4 largest L1 weights (C = 1) and example molecules that switch the bit on:", [(int(b), round(float(w1[b]), 2), sorted(env_of_bit.get(int(b), []))[:3]) for b in top])
assert l1.iloc[0, 1] == 0 and (np.diff(l1.iloc[:, 1]) >= 0).all() and l1.loc[l1.C == 1, "non-zero weights (of 1024)"].item() < 30
fig, ax = plt.subplots(1, 2, figsize=(10, 3.7)); ax[0].semilogx(reg.C, reg["fit AUC"], "o-", color=COLORS["orange"], label="fit"); ax[0].semilogx(reg.C, reg["validation AUC"], "s-", color=COLORS["blue"], label="validation (15 molecules)"); ax[0].semilogx(reg.C, reg["test AUC"], "^-", color=COLORS["green"], label="test (19 molecules)")
ax[0].set(xlabel="C (weak penalty on the right)", ylabel="AUC", title="(a) L2 penalty"); ax[0].legend(fontsize=8); ax[1].semilogx(l1.C, l1.iloc[:, 1], "o-", color=COLORS["red"]); ax[1].set(xlabel="C", ylabel="non-zero weights of 1024", title="(b) L1 penalty: sparsity"); plt.tight_layout(); plt.show()

## 8. A shared multitask network
The notebook fits each task separately and notes that a real multitask network *shares* its hidden layer: one network, one output per task, trained on the sum of the masked losses. We write it: a hidden layer of 24 ReLU units, two outputs, masked binary cross-entropy, a small $L_2$ penalty and Adam. We check the gradient of the masked loss by finite differences, check that flipping a *missing* label does not change the loss, and compare with separate networks and with the logistic regression over 20 random splits.
**Stop and predict:** does sharing a hidden layer help a task with few positives?

In [ ]:
def init_nn(p, h, t, seed): r = np.random.default_rng(seed); return {"W1": r.normal(0, np.sqrt(2 / p), (p, h)), "b1": np.zeros(h), "W2": r.normal(0, np.sqrt(1 / h), (h, t)), "b2": np.zeros(t)}
def nn_forward(P, Xa): Hh = np.maximum(0, Xa @ P["W1"] + P["b1"]); return Hh, Hh @ P["W2"] + P["b2"]
def masked_bce(P, Xa, Yt, Wk, l2=1e-4):
    _, Z = nn_forward(P, Xa); p = 1 / (1 + np.exp(-Z)); L = -(Yt * np.log(p + 1e-12) + (1 - Yt) * np.log(1 - p + 1e-12)); return (L * Wk).sum() / Wk.sum() + 0.5 * l2 * sum((P[k] ** 2).sum() for k in ("W1", "W2"))
def nn_grads(P, Xa, Yt, Wk, l2=1e-4):
    Hh, Z = nn_forward(P, Xa); p = 1 / (1 + np.exp(-Z)); dZ = (p - Yt) * Wk / Wk.sum(); g = {"W2": Hh.T @ dZ + l2 * P["W2"], "b2": dZ.sum(0)}; dH = (dZ @ P["W2"].T) * (Hh > 0); g["W1"] = Xa.T @ dH + l2 * P["W1"]; g["b1"] = dH.sum(0); return g
def train_nn(Xa, Yt, Wk, h=24, epochs=300, lr=0.01, seed=0):
    P = init_nn(Xa.shape[1], h, Yt.shape[1], seed); m = {k: np.zeros_like(v) for k, v in P.items()}; v = {k: np.zeros_like(x) for k, x in P.items()}
    for t in range(1, epochs + 1):
        g = nn_grads(P, Xa, Yt, Wk)
        for k in P: m[k] = 0.9 * m[k] + 0.1 * g[k]; v[k] = 0.999 * v[k] + 0.001 * g[k] ** 2; P[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)
    return P
Yt_all = np.nan_to_num(Y); P0 = init_nn(1024, 6, 2, 0); g0 = nn_grads(P0, X[idx_fit], Yt_all[idx_fit], Wm[idx_fit]); worst = 0.0; r_ = np.random.default_rng(1)
for k in P0:
    for _ in range(5):
        ix = tuple(r_.integers(0, s) for s in P0[k].shape); a_, b_ = ({q: z.copy() for q, z in P0.items()} for _ in range(2)); a_[k][ix] += 1e-6; b_[k][ix] -= 1e-6
        worst = max(worst, abs((masked_bce(a_, X[idx_fit], Yt_all[idx_fit], Wm[idx_fit]) - masked_bce(b_, X[idx_fit], Yt_all[idx_fit], Wm[idx_fit])) / 2e-6 - g0[k][ix]))
Yf = Yt_all[idx_fit].copy(); mi = np.argwhere(Wm[idx_fit] == 0)[0]; Yf2 = Yf.copy(); Yf2[mi[0], mi[1]] = 1 - Yf2[mi[0], mi[1]]
same_loss = masked_bce(P0, X[idx_fit], Yf, Wm[idx_fit]) == masked_bce(P0, X[idx_fit], Yf2, Wm[idx_fit])
print(f"gradient of the masked loss against finite differences (20 random weights): largest difference {worst:.1e} | the loss does not change when a MISSING label is flipped: {same_loss} ({int((Wm[idx_fit] == 0).sum())} labels are missing in the 42 training molecules)")
assert worst < 1e-7 and same_loss

In [ ]:
t0 = time.perf_counter(); rg = np.random.default_rng(0); rows = []
for s in range(20):
    p_ = rg.permutation(N); tr, tes = p_[19:], p_[:19]; Psh = train_nn(X[tr], Yt_all[tr], Wm[tr], seed=s); out = []
    for j in range(2):
        te_k = tes[Wm[tes, j] == 1]; fit_k = tr[Wm[tr, j] == 1]
        if len(np.unique(Y[te_k, j])) < 2: out += [np.nan] * 3; continue
        Pj = train_nn(X[fit_k], Yt_all[fit_k][:, [j]], Wm[fit_k][:, [j]], seed=s); w_ = logreg_l2(X[fit_k], Y[fit_k, j])
        out += [auc(Y[te_k, j], nn_forward(Psh, X[te_k])[1][:, j]), auc(Y[te_k, j], nn_forward(Pj, X[te_k])[1][:, 0]), auc(Y[te_k, j], score(X[te_k], w_))]
    rows.append(out)
rows = np.array(rows); names = ["shared network", "separate networks", "logistic regression"]; tab = []
for j, t in enumerate(("lipophilic", "hbond")):
    for m_, nm in enumerate(names):
        v = rows[:, 3 * j + m_]; v = v[np.isfinite(v)]; tab.append((t, nm, len(v), v.mean(), v.std() / math.sqrt(len(v))))
mt = pd.DataFrame(tab, columns=["task", "model", "splits with both classes", "mean test AUC", "standard error"]); print(mt.round(3).to_string(index=False)); print(f"[{time.perf_counter() - t0:.0f} s]")
dsh = rows[:, 3] - rows[:, 4]; dsh = dsh[np.isfinite(dsh)]; print(f"hbond, shared minus separate networks: {dsh.mean():+.3f} +- {dsh.std() / math.sqrt(len(dsh)):.3f} (paired differences over {len(dsh)} splits)")
assert (mt["mean test AUC"] > 0.5).all()

## 9. Your turn
**Exercise 1.** Use radius 1 and radius 3 instead of 2. How do the number of environments, the collisions and the test AUC change?
**Exercise 2.** Make the fingerprint 4,096 bits long. How do the collisions and the spread of the AUC over hash functions change?
**Exercise 3.** Add the count of each environment instead of a 0/1 bit. Can the model now tell heptane from octane?
**Exercise 4.** Put the descriptor LogP, scaled, as a second feature next to the fingerprint. What happens to the test AUC of the lipophilic task?
**Exercise 5.** Replace the random test set by the 19 heaviest molecules for both tasks and repeat Section 7. How do the choices of $C$ change?

## 10. Self-check

In [ ]:
assert (d_mw < 0.02).all() and [len(e) for e in envs[:3]] == [1, 2, 4] and pairs[0][0] == 1.0 and abs(masked - 0.36355) < 1e-5
assert ((df.LogP.values >= 1.5) == (lip == 1)).all() and ((df.HBA.values[known] >= 2) == (hb[known] == 1)).all() and np.allclose(cls.loc[[1, 2, 4, 5], "majority accuracy"], [0.8, 0.789, 0.857, 0.875], atol=1e-3)
assert worst < 1e-7 and same_loss and np.nanstd(R[:, 0]) > 0.05
print("All checks passed: you reproduced the numbers in the post.")

# Part II. Cross-check with RDKit and scikit-learn
> **Read this first.** Part II compares the functions of Part I with the libraries. With **scikit-learn** (preinstalled in Colab): our split, our logistic regressions ($L_2$ and $L_1$) and our AUC. With **RDKit** (installed by the cell if it is missing): molecular weights, the descriptors behind the labels (LogP, hydrogen-bond donors and acceptors), the number of active bits of every molecule (RDKit's fingerprint without folding against ours), the Tanimoto pairs, and the whole table of the instructor notebook (the AUCs of the logistic regression computed with RDKit's own 1,024-bit fingerprints). Each stage is guarded; the last cell lists the checks that passed.

In [ ]:
HAVE_SK = importlib.util.find_spec("sklearn") is not None; checks = []
def check(label, ok): checks.append((label, bool(ok))); print(("   ok   " if ok else "  FAIL  ") + label)
if HAVE_SK:
    import warnings; warnings.filterwarnings("ignore"); import sklearn
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    print("scikit-learn", sklearn.__version__)
    try:
        a_, b_ = train_test_split(np.arange(N), test_size=.25, random_state=32, shuffle=True); c_, d_ = train_test_split(a_, test_size=.25, random_state=32, shuffle=True)
        check("the split (42 / 15 / 19) equals train_test_split with the seed 32", np.array_equal(a_, idx_train) and np.array_equal(b_, idx_test) and np.array_equal(c_, idx_fit) and np.array_equal(d_, idx_valid))
        w2 = logreg_l2(X[idx_fit], Y[idx_fit, 0], 1.0); sk2 = LogisticRegression(C=1.0, max_iter=10000, tol=1e-10).fit(X[idx_fit], Y[idx_fit, 0].astype(int)); dw = np.abs(w2[:-1] - sk2.coef_[0]).max()
        check(f"L2 logistic regression: our coefficients equal scikit-learn's (largest difference {dw:.1e})", dw < 1e-5 and abs(w2[-1] - sk2.intercept_[0]) < 1e-5)
        w1_ = logreg_l1(X[idx_fit], Y[idx_fit, 0], 1.0, iters=20000); sk1 = LogisticRegression(penalty="l1", solver="saga", C=1.0, tol=1e-10, max_iter=200000).fit(X[idx_fit], Y[idx_fit, 0].astype(int)); d1 = np.abs(w1_[:-1] - sk1.coef_[0]).max()
        check(f"L1 logistic regression: our FISTA solution equals scikit-learn's saga (largest coefficient difference {d1:.1e}, {(np.abs(w1_[:-1]) > 1e-6).sum()} non-zero weights)", d1 < 1e-3 and (np.abs(w1_[:-1]) > 1e-6).sum() == (np.abs(sk1.coef_) > 1e-6).sum())
        s_ = score(X[idx_test], w2); check("AUC equals roc_auc_score", abs(auc(Y[idx_test, 0], s_) - roc_auc_score(Y[idx_test, 0], s_)) < 1e-12)
    except Exception as e: print("stage could not be completed:", type(e).__name__, e)
else: print("scikit-learn is not available: install it and re-run this cell.")

In [ ]:
HAVE_RD = importlib.util.find_spec("rdkit") is not None
if not HAVE_RD and HAVE_SK:
    try:
        import subprocess, sys; subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit"]); HAVE_RD = importlib.util.find_spec("rdkit") is not None
    except Exception as e: print("RDKit could not be installed:", type(e).__name__, e)
if HAVE_RD:
    try:
        from rdkit import Chem, rdBase
        from rdkit.Chem import Descriptors, Lipinski, Crippen, rdFingerprintGenerator, DataStructs
        print("RDKit", rdBase.rdkitVersion); mols = [Chem.MolFromSmiles(s) for s in sm]
        check("molecular weights: RDKit equals the CSV (largest difference below 0.01)", np.abs(np.array([Descriptors.MolWt(m) for m in mols]) - df.MolWt.values).max() < 0.01)
        d_lp = np.abs(np.array([Crippen.MolLogP(m) for m in mols]) - df.LogP.values).max(); n_hd = int((np.array([Lipinski.NumHDonors(m) for m in mols]) != df.HBD.values).sum()); n_ha = int((np.array([Lipinski.NumHAcceptors(m) for m in mols]) != df.HBA.values).sum())
        print(f"descriptors against the CSV: largest LogP difference {d_lp:.1e}; molecules with a different number of donors: {n_hd}, of acceptors: {n_ha}")
        check("LogP, hydrogen-bond donors and acceptors: RDKit's descriptors equal the CSV columns (the labels are made from them)", d_lp < 1e-3 and n_hd == 0 and n_ha == 0)
        gen_u = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2 ** 24); n_rd = np.array([len(list(gen_u.GetSparseFingerprint(m).GetOnBits())) for m in mols]); n_me = np.array([len(e) for e in envs])
        print("active bits without folding, RDKit against ours: equal for", int((n_rd == n_me).sum()), "of", N, "molecules; first three:", n_rd[:3].tolist(), "(ours", n_me[:3].tolist(), ")")
        check("the number of active bits of every molecule equals ours (our parser, hydrogens and duplicate rule match RDKit's)", np.array_equal(n_rd, n_me))
        gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024); fp_rd = [gen.GetFingerprint(m) for m in mols]
        pr_rd = sorted(((DataStructs.TanimotoSimilarity(fp_rd[i], fp_rd[j]), sm[i], sm[j]) for i in range(N) for j in range(i + 1, N)), reverse=True)[:3]
        check("the three most similar pairs equal ours (heptane-octane 1.000, hexane-octane and hexane-heptane 0.875)", [(round(a, 3), b, c) for a, b, c in pr_rd] == [(round(a, 3), b, c) for a, b, c in pairs[:3]])
        Xr = np.stack([gen.GetFingerprintAsNumPy(m).astype(float) for m in mols]); print("folded RDKit fingerprints:", Xr.shape, "| bits on in the first molecule:", int(Xr[0].sum()))
        rd_auc = {}
        for j, t in enumerate(("lipophilic", "hbond")):
            fit = idx_fit[Wm[idx_fit, j] == 1]; clf = LogisticRegression(C=1.0, max_iter=1000, random_state=32).fit(Xr[fit], Y[fit, j].astype(int))
            for nm, ix in (("validation", idx_valid), ("test", idx_test)): kk = ix[Wm[ix, j] == 1]; rd_auc[(t, nm)] = roc_auc_score(Y[kk, j], clf.predict_proba(Xr[kk])[:, 1])
        print("logistic regression on RDKit's own 1024-bit fingerprints:", {k_: round(v, 3) for k_, v in rd_auc.items()}, "| the instructor notebook: lipophilic 0.972 / 0.817, hbond 1.000 / 0.893")
        check("the instructor notebook's logistic-regression AUCs are reproduced (within 0.002)", all(abs(rd_auc[k_] - nbk[k_]) < 0.002 for k_ in nbk))
    except Exception as e: print("RDKit stage could not be completed:", type(e).__name__, e)
else: print("RDKit is not available: the RDKit checks were skipped.")
if checks:
    ok = [lab for lab, p in checks if p]; bad = [lab for lab, p in checks if not p]; print(f"\nPart II: {len(ok)} of {len(checks)} checks passed")
    for lab in bad: print("  not passed:", lab)

### What to look for
* **The parser and the duplicate rule.** If the number of active bits of every molecule equals RDKit's, the three ingredients that decide it (hydrogens, ring membership and the duplicate rule) are the same; the bit *positions* differ, because the hash functions differ.
* **The notebook's table.** With RDKit's own fingerprints the logistic regression gives the instructor notebook's AUCs, which are exact fractions of the pairs in a test set of 4 positives or 2 positives.
* **Folding.** The AUC of one split moves by a pair or two when the hash collisions change (Section 6): compare it with the AUCs of Part I.